# Human-in-the-Loop Contract Reviewer (LangGraph)

This notebook builds a LangGraph workflow for contract review with five explicit steps:
1. Retrieve relevant policy context
2. Analyze risky clauses
3. Suggest edits
4. Pause for human review (HITL)
5. Produce final output


## Why This Workflow Pattern Matters

Contracts require legal judgement. A pure automation pipeline is risky.
LangGraph lets us keep speed from automation while adding safety through a mandatory human checkpoint.


## State Graph Explained Clearly

In LangGraph, each node reads and writes a shared typed state object.
Edges define execution order.

```text
START
  -> retrieve_context
  -> analyze_contract
  -> suggest_edits
  -> [PAUSE before final_output for human review]
  -> final_output
  -> END
```

Key HITL mechanism:
- `interrupt_before=['final_output']` pauses execution right before final output generation.
- Human feedback is injected with `update_state(...)`.
- `invoke(None, config)` resumes from checkpoint.


## Step 1 - Install/Import Dependencies

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph'])
print('langgraph installed')


langgraph installed


In [2]:
from typing import List, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

print('Imports successful')


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports successful


## Step 2 - Define State

In [3]:
class ContractReviewState(TypedDict):
    contract_id: str
    contract_text: str
    retrieved_context: List[str]
    risk_findings: List[str]
    suggested_edits: List[str]
    human_decision: str
    human_notes: str
    final_output: str

print('State fields: contract_id, contract_text, retrieved_context, risk_findings, suggested_edits, human_decision, human_notes, final_output')


State fields: contract_id, contract_text, retrieved_context, risk_findings, suggested_edits, human_decision, human_notes, final_output


## Step 3 - Sample Contracts and Policy Knowledge Base

In [4]:
CONTRACT_DB = {
    'c-001': '''MASTER SERVICES AGREEMENT
1. Term: 36 months, early termination fee equals 50% of remaining value.
2. Payment: 15,000 USD monthly, late fee 18% annual, services can be suspended after 5 days.
3. Liability: Provider liability capped at fees paid in the last 1 month.
4. IP Rights: All deliverables belong exclusively to Provider.
5. Data: Provider may retain data for 7 years and use anonymized data for marketing.
6. Governing Law: Cayman Islands with mandatory arbitration in Cayman Islands.
''',
    'c-002': '''SOFTWARE LICENSE AGREEMENT
1. Auto-renewal every 12 months unless canceled 90 days before renewal date.
2. Unilateral price increase rights by vendor with 15 days notice.
3. Customer indemnifies vendor for all claims including vendor negligence.
4. Vendor may audit customer systems at any time without notice.
''',
}

POLICY_KB = [
    'Termination fees above 20% of remaining value are usually considered aggressive and should be negotiated down.',
    'IP ownership should typically transfer to client for custom-paid deliverables.',
    'Liability caps tied to one month of fees are often too low; 12 months is more balanced.',
    'Cross-border arbitration venues should align with operational jurisdiction when possible.',
    'Data retention should be limited to legal necessity and marketing use should require explicit opt-in.',
]

print('Contracts:', list(CONTRACT_DB.keys()))
print('Policy items:', len(POLICY_KB))


Contracts: ['c-001', 'c-002']
Policy items: 5


## Step 4 - Build Graph Nodes

In [5]:
def retrieve_context(state: ContractReviewState) -> dict:
    text = state['contract_text'].lower()
    matched = []

    if 'termination fee' in text or 'termination' in text:
        matched.append(POLICY_KB[0])
    if 'ip rights' in text or 'deliverables' in text:
        matched.append(POLICY_KB[1])
    if 'liability' in text:
        matched.append(POLICY_KB[2])
    if 'governing law' in text or 'arbitration' in text:
        matched.append(POLICY_KB[3])
    if 'data' in text or 'retention' in text or 'marketing' in text:
        matched.append(POLICY_KB[4])

    if not matched:
        matched = ['No direct policy match found. Escalate for manual legal review.']

    print('Node retrieve_context: collected', len(matched), 'policy references')
    return {'retrieved_context': matched}


In [6]:
def analyze_contract(state: ContractReviewState) -> dict:
    text = state['contract_text'].lower()
    findings = []

    if '50% of remaining value' in text:
        findings.append('High risk: termination fee at 50% of remaining contract value.')
    if '18% annual' in text:
        findings.append('Medium risk: late-payment interest of 18% annual may be excessive.')
    if 'last 1 month' in text:
        findings.append('High risk: liability cap at 1 month of fees is heavily one-sided.')
    if 'deliverables belong exclusively to provider' in text:
        findings.append('Critical risk: IP ownership retained by provider for custom work.')
    if '7 years' in text or 'marketing' in text:
        findings.append('Medium risk: broad data retention/usage language may breach data minimization expectations.')
    if 'cayman islands' in text:
        findings.append('Medium risk: offshore governing law and arbitration venue may increase legal cost.')
    if 'vendor negligence' in text:
        findings.append('High risk: indemnity includes vendor negligence.')
    if 'price increase' in text:
        findings.append('High risk: unilateral price increase rights without negotiated guardrails.')

    if not findings:
        findings.append('No major risks found by heuristic checks; still require legal review.')

    print('Node analyze_contract: found', len(findings), 'risk items')
    return {'risk_findings': findings}


In [7]:
def suggest_edits(state: ContractReviewState) -> dict:
    edits = []

    for finding in state['risk_findings']:
        if 'termination fee' in finding:
            edits.append('Edit: reduce termination fee to <=20% of remaining value or cap at fixed amount.')
        elif '18% annual' in finding:
            edits.append('Edit: lower late fee to reasonable market range (e.g., 8-12%) and remove compounding.')
        elif 'liability cap' in finding:
            edits.append('Edit: increase liability cap to at least 12 months of fees; carve out gross negligence and confidentiality breaches.')
        elif 'IP ownership' in finding:
            edits.append('Edit: transfer ownership of custom deliverables to client upon full payment.')
        elif 'data retention/usage' in finding:
            edits.append('Edit: limit retention duration and require explicit written consent for any marketing use.')
        elif 'governing law' in finding:
            edits.append('Edit: move governing law/arbitration to client jurisdiction or neutral venue.')
        elif 'vendor negligence' in finding:
            edits.append('Edit: remove indemnification for vendor negligence and narrow indemnity scope.')
        elif 'price increase' in finding:
            edits.append('Edit: require mutual agreement for price changes or cap annual increase with notice period.')
        else:
            edits.append('Edit: request legal clarification and add balanced language.')

    print('Node suggest_edits: proposed', len(edits), 'edits')
    return {'suggested_edits': edits}


In [8]:
def final_output(state: ContractReviewState) -> dict:
    lines = []
    lines.append('FINAL CONTRACT REVIEW SUMMARY')
    lines.append('Contract ID: ' + state['contract_id'])
    lines.append('Human decision: ' + state['human_decision'])
    lines.append('')
    lines.append('Retrieved policy context:')
    for item in state['retrieved_context']:
        lines.append('- ' + item)
    lines.append('')
    lines.append('Risk findings:')
    for item in state['risk_findings']:
        lines.append('- ' + item)
    lines.append('')
    lines.append('Suggested edits:')
    for item in state['suggested_edits']:
        lines.append('- ' + item)
    lines.append('')
    lines.append('Human notes:')
    lines.append(state['human_notes'])

    rendered = '\n'.join(lines)
    print('Node final_output: summary assembled')
    return {'final_output': rendered}


## Step 5 - Build and Compile the LangGraph Workflow

In [9]:
builder = StateGraph(ContractReviewState)
builder.add_node('retrieve_context', retrieve_context)
builder.add_node('analyze_contract', analyze_contract)
builder.add_node('suggest_edits', suggest_edits)
builder.add_node('final_output', final_output)

builder.add_edge(START, 'retrieve_context')
builder.add_edge('retrieve_context', 'analyze_contract')
builder.add_edge('analyze_contract', 'suggest_edits')
builder.add_edge('suggest_edits', 'final_output')
builder.add_edge('final_output', END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory, interrupt_before=['final_output'])

print('Graph compiled with HITL pause before final_output')


Graph compiled with HITL pause before final_output


## Step 6 - Run Phase 1 (Retrieve -> Analyze -> Suggest -> Pause)

In [10]:
config = {'configurable': {'thread_id': 'contract-hitl-31'}}
initial_state = {
    'contract_id': 'c-001',
    'contract_text': CONTRACT_DB['c-001'],
    'retrieved_context': [],
    'risk_findings': [],
    'suggested_edits': [],
    'human_decision': '',
    'human_notes': '',
    'final_output': '',
}

phase1 = app.invoke(initial_state, config)

print('Workflow paused for human review before final_output')
print('Retrieved context items:', len(phase1['retrieved_context']))
print('Risk findings:', len(phase1['risk_findings']))
print('Suggested edits:', len(phase1['suggested_edits']))


Node retrieve_context: collected 5 policy references
Node analyze_contract: found 6 risk items
Node suggest_edits: proposed 6 edits
Workflow paused for human review before final_output
Retrieved context items: 5
Risk findings: 6
Suggested edits: 6


## Step 7 - Human Review at the Pause Point

In [11]:
print('Suggested edits for reviewer:')
for i, item in enumerate(phase1['suggested_edits'], start=1):
    print(str(i) + '. ' + item)

human_decision = 'approved_with_changes'
human_notes = (
    'Prioritize IP ownership and liability cap changes as non-negotiable. '
    'Termination fee reduction is mandatory; governing law change preferred if possible.'
)

print('Human decision:', human_decision)
print('Human notes:', human_notes)


Suggested edits for reviewer:
1. Edit: reduce termination fee to <=20% of remaining value or cap at fixed amount.
2. Edit: lower late fee to reasonable market range (e.g., 8-12%) and remove compounding.
3. Edit: increase liability cap to at least 12 months of fees; carve out gross negligence and confidentiality breaches.
4. Edit: transfer ownership of custom deliverables to client upon full payment.
5. Edit: limit retention duration and require explicit written consent for any marketing use.
6. Edit: move governing law/arbitration to client jurisdiction or neutral venue.
Human decision: approved_with_changes
Human notes: Prioritize IP ownership and liability cap changes as non-negotiable. Termination fee reduction is mandatory; governing law change preferred if possible.


## Step 8 - Resume Graph and Produce Final Output

In [12]:
app.update_state(config, {'human_decision': human_decision, 'human_notes': human_notes})
phase2 = app.invoke(None, config)

print('Workflow resumed and completed')
print('\n' + phase2['final_output'])


Node final_output: summary assembled
Workflow resumed and completed

FINAL CONTRACT REVIEW SUMMARY
Contract ID: c-001
Human decision: approved_with_changes

Retrieved policy context:
- Termination fees above 20% of remaining value are usually considered aggressive and should be negotiated down.
- IP ownership should typically transfer to client for custom-paid deliverables.
- Liability caps tied to one month of fees are often too low; 12 months is more balanced.
- Cross-border arbitration venues should align with operational jurisdiction when possible.
- Data retention should be limited to legal necessity and marketing use should require explicit opt-in.

Risk findings:
- High risk: termination fee at 50% of remaining contract value.
- Medium risk: late-payment interest of 18% annual may be excessive.
- High risk: liability cap at 1 month of fees is heavily one-sided.
- Critical risk: IP ownership retained by provider for custom work.
- Medium risk: broad data retention/usage language 

## Step 9 - Inspect Checkpointed State History

In [13]:
history = list(app.get_state_history(config))
print('Checkpoint count:', len(history))
for idx, snapshot in enumerate(history):
    src = 'init'
    if snapshot.metadata and 'source' in snapshot.metadata:
        src = snapshot.metadata['source']
    keys = [k for k, v in snapshot.values.items() if v]
    print(f'Checkpoint {idx}: source={src}, populated_keys={keys}')


Checkpoint count: 7
Checkpoint 0: source=loop, populated_keys=['contract_id', 'contract_text', 'retrieved_context', 'risk_findings', 'suggested_edits', 'human_decision', 'human_notes', 'final_output']
Checkpoint 1: source=update, populated_keys=['contract_id', 'contract_text', 'retrieved_context', 'risk_findings', 'suggested_edits', 'human_decision', 'human_notes']
Checkpoint 2: source=loop, populated_keys=['contract_id', 'contract_text', 'retrieved_context', 'risk_findings', 'suggested_edits']
Checkpoint 3: source=loop, populated_keys=['contract_id', 'contract_text', 'retrieved_context', 'risk_findings']
Checkpoint 4: source=loop, populated_keys=['contract_id', 'contract_text', 'retrieved_context']
Checkpoint 5: source=loop, populated_keys=['contract_id', 'contract_text']
Checkpoint 6: source=input, populated_keys=[]


## Explanation Recap: How the State Graph Works

- `ContractReviewState` is the single source of truth that all nodes read/write.
- Retrieval node enriches the state with policy context.
- Analysis node writes structured risk findings.
- Suggestion node transforms findings into negotiable edit proposals.
- Graph pauses before final output, enabling human validation and risk control.
- Resume call completes final synthesis with human decisions injected into state.


## Limitations and Next Improvements

1. Current analysis is heuristic-based, not LLM semantic reasoning.
2. Retrieval is keyword matching; replace with vector search for larger policy corpora.
3. Add rejection loop (if `human_decision='reject'`, route back to `suggest_edits`).
4. Add audit logging and versioned policy bundles for compliance workflows.
